# Эксперимент 2: сокращение дисперсии и линейная сходимость

Цель: сравнить SGD, полный GD и SVRG на хорошо обусловленной задаче с заметной L2-регуляризацией. Для графика используется `log(F(x_k) - F*)`, где `F*` найден L-BFGS.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from src.experiments import add_residual, find_reference_minimum, make_team_oracles, plot_histories
from src.optimization import gradient_descent, sgd, svrg

FIGS = ROOT / 'figs'
FIGS.mkdir(exist_ok=True)

In [ ]:
oracle = make_team_oracles(m=2000, n=30, regcoef=5e-2, random_state=7)['Log-Cosh']
x0 = np.zeros(oracle.n)
x_star, f_star, opt_result = find_reference_minimum(oracle, x0)
print({'F*': f_star, 'lbfgs_success': opt_result.success, 'grad_norm*': np.linalg.norm(oracle.grad(x_star))})

In [ ]:
histories = {}
_, _, h = gradient_descent(oracle, x0, step_size=0.25, max_epoch=60, trace=True)
histories['GD'] = add_residual(h, f_star)

_, _, h = sgd(
    oracle, x0, batch_size=32, max_epoch=60,
    lr_schedule='inverse_sqrt', lr_params={'alpha_0': 0.20}, trace=True, random_state=2
)
histories['SGD inverse sqrt'] = add_residual(h, f_star)

_, _, h = svrg(
    oracle, x0, step_size=0.20, batch_size=32, max_epoch=60,
    inner_epochs=1, trace=True, random_state=2
)
histories['SVRG'] = add_residual(h, f_star)

plot_histories(histories, y_key='residual', logy=True, title='Log-Cosh: невязка F(x)-F*')
plt.savefig(FIGS / '02_variance_reduction_log_residual.png', dpi=200, bbox_inches='tight')

На полулогарифмическом графике линейная сходимость выглядит как почти прямая линия. У SVRG каждая внешняя итерация включает стоимость полного градиента, то есть одну эффективную эпоху до внутренних шагов.